# 🚀 Text-Based Adventure Game

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lesson 3**

| | |
|---|---|
| **Project** | Text-Based Adventure Game (portfolio project 4 of 9) |
| **You will practise** | nested dictionaries, lists of items, records, functions, `while` loops, edge cases |
| **Time** | about 60 minutes |

You are trapped on a damaged space station with a limited supply of oxygen. Explore the rooms, find the keycard, grab a spare oxygen tank and reach the escape pod before you run out of air. You will build the **world** from nested dictionaries, the **player** from a record and the **engine** from small functions.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.
5. **Demo mode:** by default the notebook "types" the answers for you, so *Run all* works without stopping. To play yourself, set `INTERACTIVE = True` in the setup cell and run it again.

In [ ]:
# ⚙️ Setup — run this cell first
# INTERACTIVE = False  -> the notebook answers for you ("demo mode"), so Run all works.
# INTERACTIVE = True   -> you type the answers yourself.
INTERACTIVE = False

def ask(prompt, demo_answer=""):
    """Works like input(). In demo mode it 'types' demo_answer for you."""
    if INTERACTIVE:
        return input(prompt)
    print(f"{prompt}{demo_answer}")
    return str(demo_answer)

print("Setup complete. Interactive mode is", "ON" if INTERACTIVE else "OFF (demo mode)")

import copy

## Part 1 · The world: a nested dictionary

Every room has a **description**, a dictionary of **exits** (direction → name of the next room) and a **list of items** lying around. All the rooms live in one big dictionary, so the whole map is *a dictionary of dictionaries*.

We wrap the map in a function, `make_world()`, so that every new game starts with a fresh copy (items you pick up in one game must be back for the next).

In [ ]:
def make_world():
    """Return a brand-new map of the space station."""
    return {
        "airlock": {
            "description": "A cold airlock. Red lights flash. A corridor leads north.",
            "exits": {"north": "corridor"},
            "items": [],
        },
        "corridor": {
            "description": "A long corridor. Doors lead west to the lab, east to the garden and north to the engine room. The airlock is south.",
            "exits": {"south": "airlock", "west": "lab", "east": "garden", "north": "engine_room"},
            "items": [],
        },
        "lab": {
            "description": "A lab full of dusty monitors. A keycard lies on a desk.",
            "exits": {"east": "corridor"},
            "items": ["keycard"],
        },
        "garden": {
            "description": "A small hydroponic garden. A spare oxygen tank hangs on the wall.",
            "exits": {"west": "corridor"},
            "items": ["oxygen tank"],
        },
        "engine_room": {
            "description": "The engine room hums. A heavy door to the north leads to the escape pod.",
            "exits": {"south": "corridor", "north": "escape_pod"},
            "items": [],
        },
        "escape_pod": {
            "description": "The escape pod! You strap in and blast away from the station.",
            "exits": {},
            "items": [],
        },
    }

# Rooms you cannot enter without a particular item
LOCKED_ROOMS = {"escape_pod": "keycard"}

world = make_world()
print("Rooms in the world:", list(world))
print("Exits of the lab:", world["lab"]["exits"])
print("Where does the corridor's east door lead?", world["corridor"]["exits"]["east"])
print("Items in the garden:", world["garden"]["items"])

## Part 2 · The player: a record

The player is a **dictionary** too: where they are, what they carry (a **list**) and how much oxygen is left.

In [ ]:
def make_player():
    return {"room": "airlock", "inventory": [], "oxygen": 8}

player = make_player()
print(player)

## Part 3 · The engine: small functions

Each function has **one job**. They receive the `world` and the `player` as parameters (rather than using global variables), so they are easy to test.

| Function | Job |
|---|---|
| `show_room` | describe where the player is |
| `get_choice` | ask for a command and split it into a *verb* and a *noun* (`go north` → `go`, `north`) |
| `move_player` | change rooms, check for locked doors, use one unit of oxygen |
| `take_item` | move an item from the room to the inventory |

In [ ]:
def show_room(world, player):
    """Describe the room the player is in."""
    room = world[player["room"]]
    print(f"\n📍 {player['room'].replace('_', ' ').title()}")
    print(room["description"])
    if room["items"]:
        print("You see:", ", ".join(room["items"]))
    print("Exits:", ", ".join(room["exits"]) if room["exits"] else "none")
    print(f"Oxygen: {player['oxygen']} moves left")

def get_choice(demo_command):
    """Ask for a command and split it into (verb, noun). 'go north' -> ('go', 'north')."""
    text = ask("\n> What do you do? ", demo_command).strip().lower()
    parts = text.split(maxsplit=1)
    if not parts:
        return "", ""
    return parts[0], (parts[1] if len(parts) > 1 else "")

def move_player(world, player, direction):
    """Try to move in a direction. Uses one unit of oxygen when it works."""
    exits = world[player["room"]]["exits"]
    if direction not in exits:
        print("You can't go that way.")
        return
    target = exits[direction]
    needed = LOCKED_ROOMS.get(target)
    if needed and needed not in player["inventory"]:
        print(f"🔒 The door is locked. You need the {needed}.")
        return
    player["room"] = target
    player["oxygen"] -= 1
    show_room(world, player)

def take_item(world, player, item):
    """Pick up an item from the current room."""
    items = world[player["room"]]["items"]
    if item not in items:
        print(f"There is no {item} here.")
        return
    items.remove(item)
    player["inventory"].append(item)
    if item == "oxygen tank":
        player["oxygen"] += 6
        print("You connect the spare tank to your suit. +6 oxygen! 🫧")
    else:
        print(f"You take the {item}.")

## Part 4 · The game loop

The loop reads a command and calls the right function. After every command it checks the **two ways the game can end**: reaching the escape pod (win) or running out of oxygen (lose).

In demo mode the player follows a script that also tries three things that should *not* work: an exit that does not exist, an item that is not there, and the locked door. Watch how the game copes with each. (To play yourself, set `INTERACTIVE = True` in the setup cell.)

In [ ]:
HELP = "Try: look, go <direction>, take <item>, inventory, quit"

def play(demo_commands):
    world = make_world()
    player = make_player()
    demo = iter(demo_commands)
    print("🚀 SPACE STATION ESCAPE 🚀")
    print("Your oxygen is running out. Find the keycard and reach the escape pod!")
    show_room(world, player)

    while True:
        verb, noun = get_choice(next(demo, "quit"))

        if verb == "quit":
            print("You give up and drift into space... 👋")
            break
        elif verb == "look":
            show_room(world, player)
        elif verb == "go":
            move_player(world, player, noun)
        elif verb == "take":
            take_item(world, player, noun)
        elif verb == "inventory":
            print("You carry:", ", ".join(player["inventory"]) or "nothing")
        else:
            print("I don't understand that.", HELP)

        if player["room"] == "escape_pod":
            print("\n🏆 YOU WIN! You escaped with", player["oxygen"], "oxygen to spare.")
            break
        if player["oxygen"] <= 0:
            print("\n💀 Your oxygen ran out. GAME OVER.")
            break

winning_script = [
    "go up",            # an exit that does not exist
    "take sandwich",    # an item that is not here
    "dance",            # a command the game does not know
    "go north", "go north",
    "go north",         # the escape pod is locked: no keycard yet!
    "go south", "go west", "take keycard", "inventory",
    "go east", "go east", "take oxygen tank",
    "go west", "go north", "go north",
]
play(winning_script)

## Part 5 · What if the player loses?

A good game has a way to lose, too. Here the player wanders back and forth and wastes all their oxygen.

In [ ]:
play(["go north", "go south"] * 4)

## ✏️ Your turn · Extend the world

Add **two new rooms and one puzzle** of your own. Ideas: an *observatory* with a telescope, a *storage bay* with a locked cabinet, a *medical bay* that gives extra oxygen, a *robot* that blocks a corridor until you give it an item.

1. Add your rooms inside `make_world()`.
2. Connect them with exits **in both directions** (if the lab has an east exit to the corridor, the corridor needs a west exit back).
3. Run the checker below to find mistakes in your map. It checks that every exit leads to a room that exists and that most exits have a way back.

In [ ]:
def check_world(world):
    """Return a list of problems found in a map."""
    problems = []
    opposite = {"north": "south", "south": "north", "east": "west", "west": "east"}
    for name, room in world.items():
        for direction, target in room["exits"].items():
            if target not in world:
                problems.append(f"{name}: the {direction} exit leads to '{target}', which does not exist")
            elif name not in world[target]["exits"].values() and target != "escape_pod":
                problems.append(f"{name}: there is no way back from {target}")
    return problems

problems = check_world(make_world())
print("✅ The map looks good!" if not problems else problems)

## 🏆 Stretch challenges

1. Add a **health** value to the player and a room with a hazard that reduces it.
2. Add a `use` command: *use keycard* opens the door instead of just carrying the card.
3. Give some items a **weight** and limit how much the player can carry (use a dictionary of item → weight).
4. **Save** the player's progress to a JSON file and load it when the game starts (Lesson 7).

## 🪞 Reflection and testing

List at least **five edge cases** you tested, and what the game did in each case. For example: typing a direction that does not exist, typing nothing at all, taking an item twice, walking into a locked door, arriving at the escape pod with no oxygen left.

*Your list:*

## 📤 How to submit

**File → Save a copy in Drive**, then **Share → Anyone with the link can view**, and send the link to your teacher. Make sure your two new rooms and your puzzle work, and that the notebook runs from the top.